# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrhman-Moubarak/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is a ranking/scoring task. The goal is an ordered priority list, telling an editor which pages to review first, so this fits the "which ones first?" pattern, where the target is a priority score and the standard metric is precision@K.


Below i loaded the dataset and checked trend_direction's categories: down, flat, stable, up, and new. These form a natural order from worse to better performance, not separate unrelated groups, which is why ranking fits better than clustering here.

In [17]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['trend_direction'].value_counts()

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

## 2. Target or proxy

The target I'm using is is_declining_label, defined as trend_direction == "down". This is a defined proxy, not an observed outcome. It's a relabeling of information already in the data, not a measurement of something that actually happened over time.

The ideal target would be an observed future outcome like given a page's performance over the last 90 days, whether it actually declines over the next 30 days. That data isn't available yet at this stage of the project, so I'm using the proxy for now, while treating its results as directional rather than a true prediction.

In [18]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['is_declining_label'].value_counts()

is_declining_label
1    16262
0    13738
Name: count, dtype: int64

## 3. Success metric

The success metric is Precision@50. With about 30,000 pages, an editor doesn't have time to review all of them, so we rank pages and flag the top 50 worth reviewing. Precision@50 measures how many of those top 50 are actually correct, which matters more than overall accuracy since the editor only ever sees the top of the list.


Below, we train a simple model, rank the test pages by predicted decline probability, and check how many of the top 50 ranked pages are actually declining. This gives a real Precision@50 number to back up the metric choice above.

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = ['content_age_days', 'avg_position', 'engagement_rate']
X = df[features]
y = df['is_declining_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]
top_50_idx = scores.argsort()[-50:]

precision_at_50 = y_test.values[top_50_idx].mean()
print(f"Precision@50: {precision_at_50:.3f}")

Precision@50: 0.780


## 4. The unit of analysis, as a real dataframe

One row in this dataset represents one page's performance over 90 days including metrics like average position, CTR, engagement rate, content age, and trend direction.

In [20]:
lane_columns = ['content_id', 'content_age_days', 'avg_position', 'ctr', 
                 'engagement_rate', 'trend_direction', 'is_declining_label']

df[lane_columns].head()

,content_id,content_age_days,avg_position,ctr,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,187,10.6,0.76,5.88,down,1
1,content_a1fb4e703a9e,445,20.3,0.05,0.00,down,1
2,content_9aa793d4d895,141,36.5,0.09,0.00,down,1
3,content_331d6c4de07b,463,6.2,0.49,1.28,stable,0
4,content_d99b7a2d90ca,263,44.0,0.13,0.00,down,1


## 5. Why ML beats a fixed rule here

A fixed rule only checks one signal and treats every page the same. The random forest trained in Section 3 combines multiple signals (content age, position, engagement) into one judgment, and its result shows the gap the official pipeline baseline scores 0.240 Precision@50, while the trained model reaches around 0.78.

In [22]:
import json

res = json.load(open("../../outputs/model_results.json"))
baseline_precision = res["baseline"]["baseline_precision_at_50"]

print(f"Baseline (output json): {baseline_precision:.3f}")
print(f"Random forest (our training result): {precision_at_50:.3f}")

Baseline (output json): 0.240
Random forest (our training result): 0.780


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.